In [ ]:
import pandas as pd

from utils.print_section import print_section

"""
05_feature_engineering
"""

In [2]:
from pathlib import Path
import pandas as pd
import sys

project_root = Path.cwd().resolve().parents[0] if Path.cwd(
).name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from utils.load_data_model import load_data_model

INPUT = "../data/processed/model_features.csv"

df = pd.read_csv(INPUT, low_memory=False)

In [11]:
from utils.print_section import  print_section
print_section("Mean")
meann = df.groupby("target")[
    [
        "profitability",
        "liquidity",
        "solvency",
        "structure",
        "log_age",
        "size",
    ]
].mean()

print(meann)

mediann =  df.groupby("target")[
    [
        "profitability",
        "liquidity",
        "solvency",
        "structure",
    ]
].median()

print_section("Median")
print(mediann)


correlationss = df[
    [
        "profitability",
        "liquidity",
        "solvency",
        "structure",
        "log_age",
        "size",
    ]
].corr()


print_section("Correlations")
print(correlationss)


# ============================================================
# Mann-Whitney U tests
# ============================================================
from scipy.stats import mannwhitneyu

print_section("Mann-Whitney U tests")

results = []

FEATURES = [
    "profitability",
    "liquidity",
    "solvency",
    "structure",
    "log_age",
    "size",
]

for feature in FEATURES:

    failed = (
        df.loc[df["target"] == 1, feature]
        .dropna()
    )

    healthy = (
        df.loc[df["target"] == 0, feature]
        .dropna()
    )

    statistic, p_value = mannwhitneyu(
        healthy,
        failed,
        alternative="two-sided"
    )

    results.append({
        "feature": feature,
        "p_value": p_value,
        "healthy_median": healthy.median(),
        "failed_median": failed.median(),
    })

results_df = pd.DataFrame(results)

print(results_df)


Mean
        profitability  liquidity  solvency  structure   log_age       size
target                                                                    
0            0.008365   4.765843  0.079154   0.740195  2.443892  12.212406
1           -0.459710   1.614071 -1.646415   0.855407  2.323365  11.196500

Median
        profitability  liquidity  solvency  structure
target                                               
0            0.036218   1.500386  0.381471   0.892597
1           -0.154000   0.655377 -0.198374   0.991753

Correlations
               profitability  liquidity  solvency  structure   log_age  \
profitability       1.000000   0.045306  0.511673  -0.022942 -0.010427   
liquidity           0.045306   1.000000  0.127501   0.036451  0.076118   
solvency            0.511673   0.127501  1.000000  -0.026824 -0.017701   
structure          -0.022942   0.036451 -0.026824   1.000000 -0.009289   
log_age            -0.010427   0.076118 -0.017701  -0.009289  1.000000   
size        

In [12]:
failed = df[
    (df["target"] == 1)
    &
    (df["dgrmovestreet"] > 0)
]

failed[
    [
        "vat",
        "bookyear",
        "dgrmovestreet",
        "jaar_van_faling"
    ]
].head(20)

,vat,bookyear,dgrmovestreet,jaar_van_faling
6587,405985481,2018,1.0,2019.0
11893,413984320,2018,1.0,2019.0
29063,421944456,2017,1.0,2018.0
35500,423998777,2017,1.0,2018.0
35628,424033223,2017,1.0,2018.0
71750,434813287,2017,1.0,2018.0
77054,436421608,2017,1.0,2018.0
93155,440933294,2017,1.0,2018.0
105139,444488048,2017,1.0,2018.0
108512,445413409,2018,1.0,2019.0


In [13]:
test_vats = [
    405985481,
    413984320,
    421944456,
]

df[
    df["vat"].isin(test_vats)
][
    [
        "vat",
        "bookyear",
        "dgrmovestreet",
        "target",
        "jaar_van_faling",
    ]
].sort_values(
    ["vat", "bookyear"]
)

,vat,bookyear,dgrmovestreet,target,jaar_van_faling
6586,405985481,2017,0.0,0,2019.0
6587,405985481,2018,1.0,1,2019.0
11892,413984320,2017,0.0,0,2019.0
11893,413984320,2018,1.0,1,2019.0
29063,421944456,2017,1.0,1,2018.0


In [15]:
(
df.groupby("target")["dgrmovestreet"]
      .mean())


target
0    0.006421
1    0.060595
Name: dgrmovestreet, dtype: float64

In [16]:

pd.crosstab(
    df["dgrmovestreet"],
    df["target"],
    normalize="index"
)


target,0,1
dgrmovestreet,,
0.0,0.997282,0.002718
1.0,0.973517,0.026483
